In [ ]:
import baltic as bt
import pandas as pd
import arviz as az

from datetime import datetime as dt
from datetime import timedelta
from dateutil.relativedelta import relativedelta
import time
from io import StringIO
import altair as alt
from zipfile import ZipFile
import math
import re
import random


import sys, subprocess, glob, os, shutil, re, importlib
from subprocess import call
import imp
from scipy.stats import gaussian_kde
import geopandas

%matplotlib inline
import matplotlib as mpl
from matplotlib import pyplot as plt
import seaborn as sns
import matplotlib.patheffects as path_effects
import matplotlib.lines as mlines
from matplotlib.font_manager import FontProperties
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.colors as clr
from matplotlib import rc
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates

import textwrap as textwrap
from textwrap import wrap

import numpy as np
from scipy.special import binom

from altair import datum
import arviz as az
from scipy.stats import gaussian_kde

alt.data_transformers.disable_max_rows()


In [ ]:
#need to convert the decimal dates back to calendar dates 
def convert_partial_year(number):

    year = int(number)
    d = timedelta(days=(number - year)*(365 + is_leap(year)))
    day_one = dt(year,1,1)
    date = d + day_one
    date = dt.strftime(date, '%Y-%m-%d')
    return date

In [ ]:
#need to convert the decimal dates back to calendar dates 
def convert_persistence(number):

    
    d = timedelta(days=(number)*(365))
    
    return d.total_seconds()


In [ ]:
def is_leap(number):
    if number == 2024:
        leap = 1
    else:
        leap = 0
    return leap


In [ ]:
def convert_format(number):
    date = dt.strptime(number, '%Y-%m-%d')
    date = date - timedelta(days=date.weekday())
    return date

In [ ]:
def convert_format_month(number):
    date = dt.strptime(number, '%Y-%m-%d')
    date = dt.strftime(date, '%Y-%m')
    return date

In [ ]:
def convert_format_month_only(number):
    date = dt.strptime(number, '%Y-%m-%d')
    date = dt.strftime(date, '%m')
    return date

In [ ]:
def convertDate(x,start,end):
    """ Converts calendar dates between given formats """
    return dt.strftime(dt.strptime(x,start),end)

In [ ]:
def decimal_to_days(decimal_value):
    days = int(decimal_value * 365)
    return days

In [ ]:
def get_taxa_lines(tree_path):    

    lines_to_write = ""
    with open(trees, 'rU') as infile:
        for line in infile: ## iterate through each line
            if 'state' not in line.lower(): #going to grab all the interesting stuff in the .trees file prior to the newick tree strings
                lines_to_write = lines_to_write + line

    return(lines_to_write)


In [ ]:
def get_burnin_value(tree_path, burnin_percent):
    with open(tree_path, 'rU') as infile:
        numtrees = 0
        for line in infile: ## iterate through each line
            if 'state' in line.lower(): #going to grab all the interesting stuff in the .trees file prior to the newick tree strings
                numtrees += 1
    
    burnin = numtrees * burnin_percent
    return(burnin)



In [ ]:
#making decimal date from string dates adapted from stackoverflow (thank you coding geniuses)
def toYearFraction(date):
    def sinceEpoch(date): # returns seconds since epoch
        return time.mktime(date.timetuple())
    s = sinceEpoch

    year = date.year
    startOfThisYear = dt(year=year, month=1, day=1)
    startOfNextYear = dt(year=year+1, month=1, day=1)

    yearElapsed = s(date) - s(startOfThisYear)
    yearDuration = s(startOfNextYear) - s(startOfThisYear)
    fraction = yearElapsed/yearDuration

    return date.year + fraction


In [ ]:
# count the migration jumps from outside LAC to inside LAC
def enumerate_migration_events(tree, traitType):
        
    output_dict = {}
    migration_events_counter = 0
    #tree_leaves = [leaf for leaf in tree.getExternal()]
    for k in tree.Objects:
        
        if traitType not in k.traits:
            trait = "root"
        else:
            trait = str(k.traits[traitType])
            
        parent_node = k.parent
        
        if traitType not in parent_node.traits:
            parent_trait = "root"
        
        # only write out migration events that are not from root to deme
        else:
            parent_trait = str(parent_node.traits[traitType])
        
        if (trait != parent_trait) and (parent_trait != "root"):
            migration_events_counter += 1

            migration_event = parent_trait + "-to-" + trait
            migration_date = parent_node.absoluteTime# + (k.absoluteTime - parent_node.absoluteTime) *random.uniform(0,1)
            parent_tmrca = parent_node.absoluteTime 
            chain_tmrca = k.absoluteTime
            size_of_chain =len([leaf for leaf in parent_node.leaves])
            leaf_list = []
            for leaf in tree.getExternal():
                if leaf.name in parent_node.leaves:
                    leaf_list.append(leaf)
            chain_latest_tip = max(x.absoluteTime for x in leaf_list)
                    
            # write to output dictionary
            output_dict[migration_events_counter] = {"type":migration_event, "date":migration_date, "parent_tmrca":parent_tmrca,
                                                     "chain_tmrca": chain_tmrca, "chain_latest_tip": chain_latest_tip, "size_of_chain":size_of_chain,
                                                     "parent_host":parent_trait,
                                                     "child_host": trait, "tree_length": sum([x.length for x in tree.Objects])}

    return(output_dict)


In [ ]:
#counts all migration events and records parent and child nodes
def run_mig_counts(all_trees, traitType):
    start_time = time.time()
    with open(all_trees, "r") as infile:

        tree_counter = 0
        trees_processed = 0
        migrations_dict = {}

        for line in infile:
            if 'tree STATE_' in line:
                tree_counter += 1

                if tree_counter > burnin:
                    temp_tree = StringIO(taxa_lines + line)
                    
                    tree = bt.loadNexus(temp_tree, absoluteTime = False)
                    tree.setAbsoluteTime(2024.9467)
                    trees_processed += 1

                    # iterate through the tree and pull out all migration events
                    migrations_dict[tree_counter] = enumerate_migration_events(tree, traitType)

    # print the amount of time this took
    total_time_seconds = time.time() - start_time
    total_time_minutes = total_time_seconds/60
    print("this took", total_time_seconds, "seconds (", total_time_minutes," minutes) to run on", trees_processed, "trees")
   
    """this will generate a multi-index dataframe from the migrations dictionary"""
    migrations_df = pd.DataFrame.from_dict({(i,j): migrations_dict[i][j] 
                           for i in migrations_dict.keys() 
                           for j in migrations_dict[i].keys()},
                       orient='index')

    migrations_df.reset_index(inplace=True)
    migrations_df.rename(columns={'level_0': 'tree_number', 'level_1': 'migration_event_number'}, inplace=True)
    
    return(migrations_df)

In [ ]:
def return_proportions_dataframe(input_df, time_unit):
    output_df = pd.DataFrame()

    
    for tree_number in set(input_df['tree_number'].tolist()):
        local_df1 = input_df[input_df['tree_number'] == tree_number]
        
        for v in list(set(input_df['type'].tolist())):
            local_df = local_df1[local_df1['type'] == v]
            total_transitions = len(local_df)

            for item in set(input_df[time_unit].tolist()):
                local_df2 = local_df[local_df[time_unit] == item]
                transitions_in_time_unit = len(local_df2)

                   
                if total_transitions != 0 :
                    prop_transitions_in_time_unit = transitions_in_time_unit/total_transitions
                else:
                    prop_transitions_in_time_unit = 0
                    


                to_add = pd.DataFrame({"migration_direction":[v],time_unit:[item],"tree_number":[tree_number], 
                                       "total_transitions":[total_transitions],
                                       "transitions_in_time_interval":[transitions_in_time_unit],
                                      "proportion_transitions_in_time_interval":[prop_transitions_in_time_unit]})
                output_df = output_df.append(to_add)
            
    return(output_df)


In [ ]:
#read in the introdcution rate
def read_in_forward_migration_rates_mascot(log_file_path):
    
    mig_rates_dict = {"sample":[]}
    
    with open(log_file_path, "r") as infile:
        line_number = 0
        for line in infile:
            #print(line_number)
            line_number += 1
            if not line.startswith("#"):  # log combiner will sometimes put the entire xml at the start of the log file
                # use the first line to find the migration rate columns
                
            # use the first line to find the migration rate columns
                if "posterior" in line:
                    all_cols = line.split("\t")
                    mig_column_indices = []   # list to store column indices
                    mig_key = {}   # dictionary to store the column index to map to column name

                    for i in range(len(all_cols)):
                        col = all_cols[i]
                        if "immigrationRate" in col:
                            mig_column_indices.append(i)

                    # make an empty dictionary to store Nes and generate dictionary to convert index to name
                    for n in mig_column_indices:
                        name = line.split("\t")[n]
                        interval = name.split(".")[1]# the syntax here is "NeLog.state01" where 0 is deme and 1 is interval 1
                        #interval = name.split(".")[2]
                       
                        mig_key[n] = name
                        mig_rates_dict[name] = []


                # read in actual parameter estimates and store in dictionary
                else:
                    sample = line.split("\t")[0]
                    mig_rates_dict["sample"].append(sample)

                    for index in mig_column_indices:
                        name = mig_key[index]
                        mig_rates_dict[name].append(line.split("\t")[index])
                    
                
                
                
    return(mig_rates_dict)

In [ ]:
# make a new dataframe that summarizes the 95% HPD estimate with mean for each deme and interval 
def generate_summary_mig_df(input_df):
    
    
    new_df = pd.DataFrame()

    for i in input_df.columns.tolist():
        if "immigrationRate." in i:
            interval = i.split(".")[1]
            local_series = input_df[i].astype('float').to_numpy()
            mean_log = local_series.mean()
            median_log = np.median(local_series)
            mean_linear = math.exp(mean_log)
            hpd_95 = az.hdi(local_series, 0.95)
            lower_hpd_log_95 = hpd_95[0]
            lower_hpd_linear_95 = math.exp(lower_hpd_log_95)
            upper_hpd_log_95 = hpd_95[1]
            upper_hpd_linear_95 = math.exp(upper_hpd_log_95)
            hpd_50 = az.hdi(local_series, 0.50)
            lower_hpd_log_50 = hpd_50[0]
            lower_hpd_linear_50 = math.exp(lower_hpd_log_50)
            upper_hpd_log_50 = hpd_50[1]
            upper_hpd_linear_50 = math.exp(upper_hpd_log_50)
            
            
            try:
                local_df = pd.DataFrame.from_dict({"interval":interval, "mean_mig_log":mean_log,"mean_mig_linear":mean_linear, 
                                                   "median_mig_log" : median_log, 
                                                   "upper_hpd_log_95":upper_hpd_log_95,"lower_hpd_log_95":[lower_hpd_log_95], 
                                                   "upper_hpd_log_50":upper_hpd_log_50,"lower_hpd_log_50":lower_hpd_log_50,
                                                   "upper_hpd_linear":upper_hpd_linear_95,"lower_hpd_linear":lower_hpd_linear_95,
                                                   "upper_hpd_linear_50":upper_hpd_linear_50, "lower_hpd_linear_50":lower_hpd_linear_50,
                                                  })
                new_df = new_df.append(local_df)
                #print(new_df)
            except:
                pass
            
   
            

            
    return(new_df)

In [ ]:
# reads in Ne from mascot log files. 
def read_in_Ne_changes_mascot(log_file_path):
    
    Ne_skyline_dict = {"sample":[]}
    
    with open(log_file_path, "r") as infile:
        line_number = 0
        for line in infile:
            line_number += 1
            if not line.startswith("#"):  # log combiner will sometimes put the entire xml at the start of the log file
                # use the first line to find the migration rate columns
                #print(line)
            # use the first line to find the migration rate columns
                if "posterior" in line:
                    all_cols = line.split("\t")
                    Ne_column_indices = []   # list to store column indices
                    Nes_key = {}   # dictionary to store the column index to map to column name

                    for i in range(len(all_cols)):
                        col = all_cols[i]
                        if "Ne." in col:
                            Ne_column_indices.append(i)

                    # make an empty dictionary to store Nes and generate dictionary to convert index to name
                    for n in Ne_column_indices:
                        name = line.split("\t")[n]
                        #deme = name.split(".")[1]# the syntax here is "Ne_region.1" where region is deme and 1 is interval 1
                        interval = name.split(".")[1]
                       
                        Nes_key[n] = name
                        Ne_skyline_dict[name] = []


                # read in actual parameter estimates and store in dictionary
                else:
                    sample = line.split("\t")[0]
                    Ne_skyline_dict["sample"].append(sample)

                    for index in Ne_column_indices:
                        name = Nes_key[index]
                        Ne_skyline_dict[name].append(line.split("\t")[index])
                    
                
    return(Ne_skyline_dict)

In [ ]:
# make a new dataframe that summarizes the 95% HPD estimate with mean for each deme and interval 
def generate_summary_ne_df(input_df):
    
    
    new_df = pd.DataFrame()

    for i in input_df.columns.tolist():
        if "Ne." in i:
            #deme = i.split(".")[1]
           # print(deme)
            #interval = 
            #print(interval)
#             if "\n" in i.split(".")[2]:
#                 interval = i.split(".")[2][0:2]
#             else:
            interval = i.split(".")[1]
           # print(interval)
            #print(interval)
            #print(i)
            #next_interval = int(interval)+1
            local_series = input_df[i].astype('float').to_numpy()
            #print(local_series)
            mean_log = local_series.mean()
            median_log = np.median(local_series)
            mean_linear = math.exp(mean_log)
            hpd_95 = az.hdi(local_series, 0.95)
            lower_hpd_log_95 = hpd_95[0]
            lower_hpd_linear_95 = math.exp(lower_hpd_log_95)
            upper_hpd_log_95 = hpd_95[1]
            upper_hpd_linear_95 = math.exp(upper_hpd_log_95)
            hpd_50 = az.hdi(local_series, 0.50)
            lower_hpd_log_50 = hpd_50[0]
            lower_hpd_linear_50 = math.exp(lower_hpd_log_50)
            upper_hpd_log_50 = hpd_50[1]
            upper_hpd_linear_50 = math.exp(upper_hpd_log_50)
            

            
            try:
                local_df = pd.DataFrame.from_dict({"interval":interval, "mean_Ne_log":mean_log,"mean_Ne_linear":mean_linear, 
                                                   "median_Ne_log" : median_log, 
                                                   "upper_hpd_log_95":upper_hpd_log_95,"lower_hpd_log_95":[lower_hpd_log_95], 
                                                   "upper_hpd_log_50":upper_hpd_log_50,"lower_hpd_log_50":lower_hpd_log_50,
                                                   "upper_hpd_linear":upper_hpd_linear_95,"lower_hpd_linear":lower_hpd_linear_95,
                                                   "upper_hpd_linear_50":upper_hpd_linear_50, "lower_hpd_linear_50":lower_hpd_linear_50,
                                                  })
                new_df = new_df.append(local_df)
                #print(new_df)
            except:
                pass
            
    return(new_df)

In [ ]:
trees =  "../multitree_coalescent/results/deduped_10_03_25.trees"
log_file_path = "../multitree_coalescent/results/multicoal_smoothed_multicoal_updated_case_prior_10_03_25_la_clusters_with_metadata_10_03_25.log"



In [ ]:
## read in trees, remove burn in
all_trees = trees
burnin_percent = 0.3
taxa_lines = get_taxa_lines(all_trees)
burnin = get_burnin_value(all_trees, burnin_percent)
print(burnin)



In [ ]:
#identify each migration jump across posterior set of trees
migrations_df = run_mig_counts(all_trees, traitType = "obs")

In [ ]:
## convert dates
migrations_df['calendar_date'] = migrations_df.date.map(convert_partial_year)
migrations_df['year-week'] = migrations_df['calendar_date'].map(convert_format)


In [ ]:
migrations_df.head()

In [ ]:
## extract out columns needed for plotting
migrations_for_plot = migrations_df.groupby(["migration_event_number"])["date",'parent_tmrca', "chain_tmrca", "chain_latest_tip", "size_of_chain"].median().reset_index()
migrations_for_plot = migrations_for_plot.sort_values(by=['date']).reset_index()

migrations_for_plot["calendar_date"] = migrations_for_plot.date.map(convert_partial_year)
migrations_for_plot['month'] = migrations_for_plot['calendar_date'].map(convert_format_month_only)

In [ ]:
migrations_for_plot.head()

In [ ]:
migrations_for_export = migrations_for_plot.copy()
migrations_for_export['length_of_chain'] = migrations_for_export.chain_latest_tip - migrations_for_export.date
migrations_for_export['length_of_chain_days'] = migrations_for_export.length_of_chain.apply(decimal_to_days)
migrations_for_export['calendar_date_of_import'] = migrations_for_export.date.map(convert_partial_year)
migrations_for_export['latest_case_of_chain_calendar_date'] = migrations_for_export.chain_latest_tip.map(convert_partial_year)
migrations_for_export= migrations_for_export[["migration_event_number", "calendar_date_of_import","latest_case_of_chain_calendar_date", "size_of_chain", "length_of_chain_days"]]
migrations_for_export.to_csv("importation_transmission_chains.csv")


In [ ]:
## extract out migration rates from log files 

migration_rates_f = read_in_forward_migration_rates_mascot(log_file_path)
mig_df_f = pd.DataFrame.from_dict(migration_rates_f)

burnin_percent = 0.3
print(len(mig_df_f))
rows_to_remove = int(len(mig_df_f)* burnin_percent)
mig_df_f = mig_df_f.iloc[rows_to_remove:]

print(len(mig_df_f))
mig_df_f = mig_df_f.reset_index()
mig_df_f.head()

In [ ]:
mig_summary = generate_summary_mig_df(mig_df_f)


In [ ]:
## format dates
test_mig = mig_summary
test_mig['days'] = (test_mig.interval.astype(int))*7.04
test_mig['date'] = dt.strptime("2024-12-12",  "%Y-%m-%d") - test_mig.days.map(timedelta)
test_mig["decimal_date"]= test_mig.date.map(toYearFraction)

In [ ]:
## read in Ne from log files
Ne_skyline = read_in_Ne_changes_mascot(log_file_path)

In [ ]:
## remove burn in

Ne_df = pd.DataFrame.from_dict(Ne_skyline)
print(len(Ne_df))
Ne_df

burnin_percent = 0.3
print(len(Ne_df))
rows_to_remove = int(len(Ne_df)* burnin_percent)
Ne_df = Ne_df.iloc[rows_to_remove:]

print(len(Ne_df))
Ne_df = Ne_df.reset_index()
Ne_df.head()

In [ ]:
## format Ne dates
ne_summary = generate_summary_ne_df(Ne_df)
test_ne = ne_summary
test_ne['days'] = (test_ne.interval.astype(int))*7
test_ne['date'] = dt.strptime("2024-12-12",  "%Y-%m-%d") - test_ne.days.map(timedelta)
test_ne["decimal_date"]= test_ne.date.map(toYearFraction)

#rolling mean
columns = ["mean_Ne_linear", "upper_hpd_linear_50","lower_hpd_linear_50", "lower_hpd_linear", "upper_hpd_linear"]
for column in columns:
    # Create a new column for the moving average
    test_ne[f'{column}_MA'] = test_ne[column].rolling(5, min_periods =1).mean()

In [ ]:
colors = ["#D0A854",
          "#2664A5",
          "#A76BB1",
          "#D07954",
          "#356D4C",
          "#B9B9B9"
         ]

In [ ]:
## plot Fig 4 subplot. 

# scaffold the plot
fig,ax = plt.subplots(figsize=(16,12),facecolor='w')
  
# set blank white face for background    
ax.set_facecolor('white')

# remove grid 
ax.grid(False)

# define cluster size categories for scatter plot
for index, values in migrations_for_plot.iterrows():
   # print(index)
    clust = index + 10
    if values.size_of_chain <2:
        col = colors[5]
    elif (values.size_of_chain >1) & (values.size_of_chain <5):
        col = colors[1]
    elif (values.size_of_chain >4) & (values.size_of_chain <11):
        col = colors[4]
    else:
        col = colors[3]
    linewidth = 3
    
    ax.scatter([values.date, values.date], [clust, clust], color=col, linewidth=linewidth)
    ax.plot([values.date, values.chain_latest_tip], [clust, clust], color=col, linewidth=1, linestyle = ":")


fc = colors[0]
ec = colors[0]

# create figure overlay for importation rate. 
ax2 = ax.twinx()

ax2.plot(test_mig.decimal_date,test_mig["mean_mig_linear"],color=fc,ls='--',lw=2)
ax2.fill_between(test_mig.decimal_date,test_mig.lower_hpd_linear_50,test_mig.upper_hpd_linear_50,alpha=0.05,facecolor=fc,edgecolor=ec,zorder=1000)
ax2.plot(test_mig.decimal_date,test_mig.lower_hpd_linear_50,color=fc,lw=1,zorder=1000)
ax2.plot(test_mig.decimal_date,test_mig.upper_hpd_linear_50,color=fc,lw=1,zorder=1000)
ax2.set_ylim(0,30)
ax2.set_ylabel('Local Importation Rate in LA County (events/lineage/year)', fontsize=16, rotation = -90, labelpad=20)


fc = colors[2]
ec = colors[2]


# create the legend
legend_list = [mlines.Line2D([0], [0], color=colors[5], lw=4, label='Singletons'),
                mlines.Line2D([0], [0], color=colors[1], lw=4, label='2-4'),
                mlines.Line2D([0], [0], color=colors[4], lw=4, label='5-9'),
                mlines.Line2D([0], [0], color=colors[3], lw=4, label='10+')]
ax.legend(handles=legend_list, title='Size of Local transmission cluster', fontsize=13, title_fontsize=13, loc='center right')

# add in vertical stripes for easier date visualization
xDates=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,13)]
xDates2=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,13)]

every=1
[ax.axvspan(bt.decimalDate(xDates2[x]),bt.decimalDate(xDates2[x])+1/float(12),facecolor='k',edgecolor='none',alpha=0.04) for x in range(0,len(xDates2),2)]
ax.set_xticks([bt.decimalDate(x)+1/24.0 for x in xDates if (int(x.split('-')[1])-1)%every==0])

ax.set_xticklabels([convertDate(x,'%Y-%m-%d','%Y') if x.split('-')[1]=='01' else convertDate(x,'%Y-%m-%d','%b') for x in xDates if (int(x.split('-')[1])-1)%every==0])
ax.tick_params(axis='x',labelsize=10,size=0)  

#ax1.xaxis.tick_bottom()
ax.yaxis.tick_left()

[ax2.spines[loc].set_visible(False) for loc in ['top','left']]
[ax.spines[loc].set_visible(False) for loc in ['top','right','left']]

ax.tick_params(axis='y',size=0)
ax.set_yticklabels([])
ax.set_ylim(-5,clust+80)
ax.set_xlim(2022,2025.1)

ax.xaxis.set_tick_params(which='both', top=False, bottom=True, labelbottom=True)
ax.yaxis.set_tick_params(which='both', right=False, left=True, labelleft=True)
#plt.savefig('../figures/mpox_la_introduction_rate.png',dpi=300,bbox_inches='tight')


In [ ]:
## Plot Figure S5A -- full fig S5 is at the bottom of this notebook. 

fig,ax = plt.subplots(figsize=(25,10),facecolor='w')
   
# set blank white face for background    
ax.set_facecolor('white')
# remove grid 
ax.grid(False)

for index, values in migrations_for_plot.iterrows():
   # print(index)
    clust = index + 10
    if values.size_of_chain <2:
        col = colors[5]
    elif (values.size_of_chain >1) & (values.size_of_chain <5):
        col = colors[1]
    elif (values.size_of_chain >4) & (values.size_of_chain <11):
        col = colors[4]
    else:
        col = colors[3]
    linewidth = 3
    persist_days = decimal_to_days(values.chain_latest_tip - values.date)
    radius = np.sqrt(values.size_of_chain/np.pi)*300.0
    
    ax.scatter(values.date,persist_days,s=600,facecolor=col,edgecolor='k',lw=2,zorder=200) ## add big circle at base of tree to indicate origin
    ax.plot([values.date, values.chain_latest_tip], [persist_days, persist_days], color=col, linewidth=1, linestyle = ":")

ax.set_ylabel('Persistence time (circles in days)', fontsize=28)
   
fc = colors[0]
ec = colors[0]
ax2 = ax.twinx()

ax2.plot(test_mig.decimal_date,test_mig["mean_mig_linear"],color=fc,ls='--',lw=2)

#ax.scatter(caseDates,[0.0]*len(caseDates),alpha=0.2,s=200,marker='|',lw=3,facecolor='k',zorder=100)
ax2.fill_between(test_mig.decimal_date,test_mig.lower_hpd_linear_50,test_mig.upper_hpd_linear_50,alpha=0.05,facecolor=fc,edgecolor=ec,zorder=1000)
ax2.plot(test_mig.decimal_date,test_mig.lower_hpd_linear_50,color=fc,lw=1,zorder=1000)
ax2.plot(test_mig.decimal_date,test_mig.upper_hpd_linear_50,color=fc,lw=1,zorder=1000)
ax2.set_ylim(0,30)



fc = colors[2]
ec = colors[2]
ax3 = ax.twinx()

ax3.plot(test_ne.decimal_date,test_ne["mean_Ne_linear_MA"],color=fc,ls='--',lw=2)

#ax.scatter(caseDates,[0.0]*len(caseDates),alpha=0.2,s=200,marker='|',lw=3,facecolor='k',zorder=100)


ax3.fill_between(test_ne.decimal_date,test_ne.lower_hpd_linear_50_MA,test_ne.upper_hpd_linear_50_MA,alpha=0.05,facecolor=fc,edgecolor=ec,zorder=1000)
ax3.plot(test_ne.decimal_date,test_ne.lower_hpd_linear_50_MA,color=fc,lw=1,zorder=1000)
ax3.plot(test_ne.decimal_date,test_ne.upper_hpd_linear_50_MA,color=fc,lw=1,zorder=1000)
ax3.set_ylim(0,30)
ax3.set_ylabel('Importation Rate (Yellow) & Effective Population Size (Purple)', fontsize=18, rotation = -90, labelpad=20)



legend_list = [mlines.Line2D([0], [0], color=colors[5], lw=4, label='Singletons'),
                mlines.Line2D([0], [0], color=colors[1], lw=4, label='2-4'),
                mlines.Line2D([0], [0], color=colors[4], lw=4, label='5-9'),
                mlines.Line2D([0], [0], color=colors[3], lw=4, label='10+')]
ax.legend(handles=legend_list, title='Size of Local transmission cluster', fontsize=16, title_fontsize=16, loc='upper right')

xDates=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,13)]
xDates2=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,13)]


every=1
[ax.axvspan(bt.decimalDate(xDates2[x]),bt.decimalDate(xDates2[x])+1/float(12),facecolor='k',edgecolor='none',alpha=0.04) for x in range(0,len(xDates2),2)]
ax.set_xticks([bt.decimalDate(x)+1/24.0 for x in xDates if (int(x.split('-')[1])-1)%every==0])

ax.set_xticklabels([convertDate(x,'%Y-%m-%d','%Y') if x.split('-')[1]=='01' else convertDate(x,'%Y-%m-%d','%b') for x in xDates if (int(x.split('-')[1])-1)%every==0])
ax.tick_params(axis='x',labelsize=15,size=0)  
ax.tick_params(axis='y',size=0, labelsize=20)
ax.yaxis.tick_left()
[ax2.spines[loc].set_visible(False) for loc in ['top',]]
[ax.spines[loc].set_visible(False) for loc in ['top','right']]
#ax.set_yticklabels([])
ax.set_ylim(0,500)
ax.set_xlim(2022,2025.1)

ax.xaxis.set_tick_params(which='both', top=False, bottom=True, labelbottom=True)
ax.yaxis.set_tick_params(which='both', right=False, left=True, labelleft=True)
#plt.savefig('../figures/mpox_la_introduction_persistence_with_ne.png',dpi=300,bbox_inches='tight')


In [ ]:
fig,ax = plt.subplots(figsize=(16,12),facecolor='w')

ins_ax = ax.inset_axes([.3, .65, .5, .3])  # [x, y, width, height] w.r.t. ax
    
# set blank white face for background    
ax.set_facecolor('white')
# remove grid 
ax.grid(False)


for index, values in migrations_for_plot.iterrows():
   # print(index)
    clust = index + 10
    if values.size_of_chain <2:
        col = colors[5]
    elif (values.size_of_chain >1) & (values.size_of_chain <5):
        col = colors[1]
    elif (values.size_of_chain >4) & (values.size_of_chain <11):
        col = colors[4]
    else:
        col = colors[3]
    linewidth = 3
    
    ax.plot([values.date, values.date], [0, values.size_of_chain], color=col, linewidth=linewidth)
    #ax.plot([values.chain_tmrca, values.chain_latest_tip], [0, clust], color=col, linewidth=1, linestyle = ":")

   # ax.plot([mrca[0], mrca[1]], [clust, clust], color=col, linewidth=linewidth)
    # add small vertical lines at the start and end of each mrca
    # ax.plot([mrca[0], mrca[0]], [clust-0.2, clust+0.2], color=col, linewidth=1)
    # ax.plot([mrca[1], mrca[1]], [clust-0.2, clust+0.2], color=col, linewidth=1)

# # set ylabel, with a long arrow at the end
#ax.set_ylabel('Importation (from earliest to latest) →', fontsize=fontsize)


    if values.date > 2022.99:
    
        ins_ax.plot([values.date, values.date], [0, values.size_of_chain], color=col, linewidth=linewidth)
#plt.xticks([]); plt.yticks([])  # strip ticks, which collide w/ main ax


legend_list = [mlines.Line2D([0], [0], color=colors[5], lw=4, label='Singletons'),
                mlines.Line2D([0], [0], color=colors[1], lw=4, label='2-4'),
                mlines.Line2D([0], [0], color=colors[4], lw=4, label='5-9'),
                mlines.Line2D([0], [0], color=colors[3], lw=4, label='10+')]
ax.legend(handles=legend_list, title='Size of Local transmission cluster', fontsize=15, title_fontsize=10, loc='center right')

xDates=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,12)]
xDates2=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,13)]


every=1
[ax.axvspan(bt.decimalDate(xDates2[x]),bt.decimalDate(xDates2[x])+1/float(12),facecolor='k',edgecolor='none',alpha=0.04) for x in range(0,len(xDates2),2)]
ax.set_xticks([bt.decimalDate(x)+1/24.0 for x in xDates if (int(x.split('-')[1])-1)%every==0])

ax.set_xticklabels([convertDate(x,'%Y-%m-%d','%Y') if x.split('-')[1]=='01' else convertDate(x,'%Y-%m-%d','%b') for x in xDates if (int(x.split('-')[1])-1)%every==0])
ax.tick_params(axis='x',labelsize=10,size=0)  

#ax1.xaxis.tick_bottom()
ax.yaxis.tick_left()

[ax.spines[loc].set_visible(False) for loc in ['top','right','left']]

ax.tick_params(axis='y',size=0)
ax.set_yticklabels([])
ax.set_ylim(0,60)
ax.set_xlim(2022,2025.1)

ax.xaxis.set_tick_params(which='both', top=False, bottom=True, labelbottom=True)
ax.yaxis.set_tick_params(which='both', right=False, left=True, labelleft=True)

In [ ]:
## create size categories
migrations_for_plot["clust_cat"] = "Singletons"
#migrations_for_plot.clust_cat[migrations_for_plot.size_of_chain <2] = 1
migrations_for_plot.clust_cat[(migrations_for_plot.size_of_chain >1) & (migrations_for_plot.size_of_chain <5)] = "2-4"
migrations_for_plot.clust_cat[(migrations_for_plot.size_of_chain >4) & (migrations_for_plot.size_of_chain <11)] = "5-9"
migrations_for_plot.clust_cat[migrations_for_plot.size_of_chain >10] = "10+"


In [ ]:
## composite figure 4
fig = plt.figure(figsize=(16,16),facecolor='w')


gs = GridSpec(2, 2, height_ratios=[2, 1], width_ratios=[1, 1], hspace=0.1)  

# Add the first subplot (main plot with all the identified introductions)
ax1 = fig.add_subplot(gs[0, :])

# set blank white face for background    
ax1.set_facecolor('white')

# remove grid 
ax1.grid(False)

# plot each introduction
for index, values in migrations_for_plot.iterrows():
   # print(index)
    clust = index + 10
    if values.size_of_chain <2:
        col = colors[5]
    elif (values.size_of_chain >1) & (values.size_of_chain <5):
        col = colors[1]
    elif (values.size_of_chain >4) & (values.size_of_chain <11):
        col = colors[4]
    else:
        col = colors[3]
    linewidth = 3
    
    ax1.scatter([values.date, values.date], [clust, clust], color=col, linewidth=linewidth)
    ax1.plot([values.date, values.chain_latest_tip], [clust, clust], color=col, linewidth=1, linestyle = ":")

fc = colors[0]
ec = colors[0]

# plot the rate of introduction into LA County
ax2 = ax1.twinx()

#first plot the mean
ax2.plot(test_mig.decimal_date,test_mig["mean_mig_linear"],color=fc,ls='--',lw=2)

# then the bounds
ax2.fill_between(test_mig.decimal_date,test_mig.lower_hpd_linear_50,test_mig.upper_hpd_linear_50,alpha=0.05,facecolor=fc,edgecolor=ec,zorder=1000)
ax2.plot(test_mig.decimal_date,test_mig.lower_hpd_linear_50,color=fc,lw=1,zorder=1000)
ax2.plot(test_mig.decimal_date,test_mig.upper_hpd_linear_50,color=fc,lw=1,zorder=1000)
ax2.set_ylim(0,30)
ax2.set_ylabel('Local Importation Rate in LA County (events/lineage/year)', fontsize=16, rotation = -90, labelpad=20)

fc = colors[2]
ec = colors[2]

legend_list = [mlines.Line2D([0], [0], color=colors[5], lw=4, label='Singletons'),
                mlines.Line2D([0], [0], color=colors[1], lw=4, label='2-4'),
                mlines.Line2D([0], [0], color=colors[4], lw=4, label='5-9'),
                mlines.Line2D([0], [0], color=colors[3], lw=4, label='10+')]
ax1.legend(handles=legend_list, title='Size of Local transmission cluster', fontsize=13, title_fontsize=13, loc='center right')

xDates=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,13)]
xDates2=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,13)]


every=1
[ax1.axvspan(bt.decimalDate(xDates2[x]),bt.decimalDate(xDates2[x])+1/float(12),facecolor='k',edgecolor='none',alpha=0.04) for x in range(0,len(xDates2),2)]
ax1.set_xticks([bt.decimalDate(x)+1/24.0 for x in xDates if (int(x.split('-')[1])-1)%every==0])

ax1.set_xticklabels([convertDate(x,'%Y-%m-%d','%Y') if x.split('-')[1]=='01' else convertDate(x,'%Y-%m-%d','%b') for x in xDates if (int(x.split('-')[1])-1)%every==0])
ax1.tick_params(axis='x',labelsize=10,size=0)  
ax1.yaxis.tick_left()

[ax2.spines[loc].set_visible(False) for loc in ['top','left']]
[ax1.spines[loc].set_visible(False) for loc in ['top','right','left']]

ax1.tick_params(axis='y',size=0)
ax1.set_yticklabels([])
ax1.set_ylim(-5,clust+80)
ax1.set_xlim(2022,2025.1)

ax1.xaxis.set_tick_params(which='both', top=False, bottom=True, labelbottom=True)
ax1.yaxis.set_tick_params(which='both', right=False, left=True, labelleft=True)

#add in panel label 
ax1.text(-0.01, 1, "A", transform=ax1.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')  # Panel label



## now add in the number of importations per month (bottom left subplot) 
ax3 = fig.add_subplot(gs[1, 0])

#no_second_year = ["09", "10", "11", "12"]
no_second_year = []
monthly_counts = migrations_for_plot[migrations_for_plot.date >2023].groupby('month').size().reset_index(name='counts')
for index, row in monthly_counts.iterrows():
    if row.month not in no_second_year:
        monthly_counts.loc[index, "counts"] = monthly_counts.loc[index, "counts"]/2
# Plot histogram
sns.barplot(x='month', y='counts', data=monthly_counts, color = "gray", ax = ax3)

# Adjust plot aesthetics
#ax3.xticks(rotation=45, ha='right', fontsize=8)  # Rotate x-axis labels
#ax3.title("Introductions per Month (since 2023)")
ax3.set_xlabel("Month of Year", fontsize=12)
ax3.set_ylabel("Introductions into LA County", fontsize=12)
ax3.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep", "Oct", "Nov", "Dec"])

#add in panel label 
ax3.text(-0.1, 1, "B", transform=ax3.transAxes, fontsize=20, fontweight='bold', va='top', ha='right') 


## plotting persistence time for each month (bottom right subplot)
ax4 = fig.add_subplot(gs[1, 1])

migrations_for_plot_2 = migrations_for_plot.sort_values(by=['month']).reset_index() ## this is done to organize the x axis

# fig,ax = plt.subplots(figsize=(25,10),facecolor='w')

sns.set_style('white')
palette = 'Set2'

# estimate persistence
migrations_for_plot_2["persist_days"] = (migrations_for_plot_2.chain_latest_tip - migrations_for_plot_2.date).apply(decimal_to_days)

#plot distribution of persistence times
sns.violinplot(x="month", y="persist_days", data=migrations_for_plot_2[migrations_for_plot_2.date >2023], order=month_order, dodge=False,
                    palette = [colors[5]] * 12 ,
                    scale="width", inner=None, cut = 0, saturation = 0, ax = ax4 )
xlim = ax4.get_xlim()
ylim = ax4.get_ylim()

#cut off half of the violin plot for the raincloud plot
for violin in ax4.collections:
    bbox = violin.get_paths()[0].get_extents()
    x0, y0, width, height = bbox.bounds
    violin.set_clip_path(plt.Rectangle((x0, y0), width/2, height, transform=ax4.transData))

#make the boxplot with the same data
sns.boxplot(x="month", y="persist_days", data=migrations_for_plot_2[migrations_for_plot_2.date >2023],order=month_order, saturation=1, showfliers=False,
            width=0.75, boxprops={'zorder': 3, 'facecolor': 'none'}, ax=ax4)
old_len_collections = len(ax4.collections)

#now add in the actual scatterpoints
sns.scatterplot(x="month", y="persist_days", hue="clust_cat", s = 150,
             alpha=0.7, linewidth = 1,edgecolor = "k", palette=["gray",  colors[3] , colors[1],colors[4],],
             data=migrations_for_plot_2[migrations_for_plot_2.date >2023], order=month_order, ax=ax4)

for dots in ax4.collections[old_len_collections:]:
    dots.set_offsets(dots.get_offsets() + np.array([0.15, 0]))
    
ax4.set_xlim(xlim)
ax4.set_ylim(ylim)

legend_list = [mlines.Line2D([], [], color=colors[5], lw=0, label='Singletons', marker = "o"),
                mlines.Line2D([0], [0], color=colors[1], lw=0, label='2-4', marker = "o"),
                mlines.Line2D([0], [0], color=colors[4], lw=0, label='5-9', marker = "o"),
                mlines.Line2D([0], [0], color=colors[3], lw=0, label='10+', marker = "o")]

ax4.legend(handles = legend_list,title='Size of Local transmission cluster', fontsize=11, title_fontsize=11, loc='upper center')
ax4.set_ylabel('Persistance of Transmission Chain \n (in days since 2023)', fontsize=12)
ax4.set_xlabel('Month of Year', fontsize = 12)
ax4.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep", "Oct", "Nov", "Dec"])

#add in panel label 
ax4.text(-0.1, 1, "C", transform=ax4.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')  


#plt.savefig('../figures/mpox_la_introduction_figure.png',dpi=300,bbox_inches='tight')


In [ ]:
##  Main fig 4. 
fig = plt.figure(figsize=(16,16), facecolor='w')
gs = GridSpec(2, 2, height_ratios=[2, 1], width_ratios=[1, 1], hspace=0.1)

## 
## Introductions
##

ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor('white')
ax1.grid(False)

# plot each introduction
for index, values in migrations_for_plot.iterrows():
    clust = index + 10
    if values.size_of_chain < 2:
        col = colors[5]
    elif (values.size_of_chain > 1) & (values.size_of_chain < 5):
        col = colors[1]
    elif (values.size_of_chain > 4) & (values.size_of_chain < 11):
        col = colors[4]
    else:
        col = colors[3]
    linewidth = 3
    ax1.scatter([values.date, values.date], [clust, clust], color=col, linewidth=linewidth)
    ax1.plot([values.date, values.chain_latest_tip], [clust, clust], color=col, linewidth=1, linestyle=":")

fc = colors[0]
ec = colors[0]

# Plot Rate of introduction into LA County (R axis)
ax2 = ax1.twinx()
ax2.plot(test_mig.decimal_date, test_mig["mean_mig_linear"], color=fc, ls='--', lw=2)
ax2.fill_between(test_mig.decimal_date, test_mig.lower_hpd_linear_50, test_mig.upper_hpd_linear_50,
                 alpha=0.05, facecolor=fc, edgecolor=ec, zorder=1000)
ax2.plot(test_mig.decimal_date, test_mig.lower_hpd_linear_50, color=fc, lw=1, zorder=1000)
ax2.plot(test_mig.decimal_date, test_mig.upper_hpd_linear_50, color=fc, lw=1, zorder=1000)
ax2.set_ylim(0, 30)
ax2.set_ylabel('Local Importation Rate in LA County (events/lineage/year)', fontsize=16, rotation=-90, labelpad=20)

fc = colors[2]
ec = colors[2]

legend_list = [
    mlines.Line2D([0], [0], color=colors[5], lw=4, label='Singletons'),
    mlines.Line2D([0], [0], color=colors[1], lw=4, label='2-4'),
    mlines.Line2D([0], [0], color=colors[4], lw=4, label='5-9'),
    mlines.Line2D([0], [0], color=colors[3], lw=4, label='10+'),
]
ax1.legend(handles=legend_list, title='Size of Local transmission cluster', fontsize=13, title_fontsize=13, loc='center right')

xDates  = ['%04d-%02d-01' % (y, m) for y in range(2022, 2025) for m in range(1, 13)]
xDates2 = ['%04d-%02d-01' % (y, m) for y in range(2022, 2025) for m in range(1, 13)]

every = 1
[ax1.axvspan(bt.decimalDate(xDates2[x]), bt.decimalDate(xDates2[x]) + 1/12.0, facecolor='k', edgecolor='none', alpha=0.04)
 for x in range(0, len(xDates2), 2)]
ax1.set_xticks([bt.decimalDate(x) + 1/24.0 for x in xDates if (int(x.split('-')[1]) - 1) % every == 0])

ax1.set_xticklabels([convertDate(x, '%Y-%m-%d', '%Y') if x.split('-')[1] == '01' else convertDate(x, '%Y-%m-%d', '%b')
                     for x in xDates if (int(x.split('-')[1]) - 1) % every == 0])
ax1.tick_params(axis='x', labelsize=10, size=0)
ax1.yaxis.tick_left()

[ax2.spines[loc].set_visible(False) for loc in ['top', 'left']]
[ax1.spines[loc].set_visible(False) for loc in ['top', 'right', 'left']]

ax1.tick_params(axis='y', size=0)
ax1.set_yticklabels([])
ax1.set_ylim(-5, clust + 80)
ax1.set_xlim(2022, 2025.1)

ax1.xaxis.set_tick_params(which='both', top=False, bottom=True, labelbottom=True)
ax1.yaxis.set_tick_params(which='both', right=False, left=True, labelleft=True)

# Panel label
ax1.text(-0.01, 1, "A", transform=ax1.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')


## 
## Introductions per month (bottom Right)
## 

ax3 = fig.add_subplot(gs[1, 0])

# Construct a complete month list 
month_order_codes = [f"{m:02d}" for m in range(1, 13)]  # "01"..."12"
months_df = pd.DataFrame({"month": month_order_codes})

monthly_counts = (
    migrations_for_plot[migrations_for_plot.date > 2023]
    .groupby("month", as_index=False)
    .size()
    .rename(columns={"size": "counts"})
)

# Merge !
monthly_counts = months_df.merge(monthly_counts, on="month", how="left")
monthly_counts["counts"] = monthly_counts["counts"].fillna(0)

# scale in case some months dont have enough data
no_second_year = [] 
mask = ~monthly_counts["month"].isin(no_second_year)
monthly_counts.loc[mask, "counts"] = monthly_counts.loc[mask, "counts"] / 2

# Plot 
sns.barplot(x="month", y="counts", data=monthly_counts, order=month_order_codes, color="gray", ax=ax3)

ax3.set_xlabel("Month of Year", fontsize=12)
ax3.set_ylabel("Introductions into LA County since 2023", fontsize=12)
ax3.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax3.text(-0.1, 1, "B", transform=ax3.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')


## 
## Persistence per month (bottom R)
## 

ax4 = fig.add_subplot(gs[1, 1])

migrations_for_plot_2 = migrations_for_plot.sort_values(by=['month']).reset_index(drop=True).copy()
sns.set_style('white')

# Estimate persistence
migrations_for_plot_2["persist_days"] = (migrations_for_plot_2.chain_latest_tip - migrations_for_plot_2.date).apply(decimal_to_days)

# Ensure month is cat
migrations_for_plot_2["month"] = pd.Categorical(migrations_for_plot_2["month"], categories=month_order_codes, ordered=True)

subset = migrations_for_plot_2[migrations_for_plot_2.date > 2023]

# violin plot.
sns.violinplot(
    x="month", y="persist_days", data=subset,
    order=month_order_codes, dodge=False,
    palette=[colors[5]] * 12, scale="width", inner=None, cut=0, saturation=0, ax=ax4
)
xlim = ax4.get_xlim()
ylim = ax4.get_ylim()

# Cut off half of the violin 
for violin in ax4.collections:
    bbox = violin.get_paths()[0].get_extents()
    x0, y0, width, height = bbox.bounds
    violin.set_clip_path(plt.Rectangle((x0, y0), width/2, height, transform=ax4.transData))

# Boxplot 
sns.boxplot(
    x="month", y="persist_days", data=subset,
    order=month_order_codes, saturation=1, showfliers=False,
    width=0.75, boxprops={'zorder': 3, 'facecolor': 'none'}, ax=ax4
)
old_len_collections = len(ax4.collections)

# Scatter! Wow this is complex. 
sns.scatterplot(
    x="month", y="persist_days", hue="clust_cat", s=150, alpha=0.7, linewidth=1, edgecolor="k",
    palette=["gray", colors[3], colors[1], colors[4]],
    data=subset, ax=ax4
)

# Jitter for visability
for dots in ax4.collections[old_len_collections:]:
    dots.set_offsets(dots.get_offsets() + np.array([0.15, 0]))

ax4.set_xlim(xlim)
ax4.set_ylim(ylim)

legend_list = [
    mlines.Line2D([], [], color=colors[5], lw=0, label='Singletons', marker="o"),
    mlines.Line2D([0], [0], color=colors[1], lw=0, label='2-4', marker="o"),
    mlines.Line2D([0], [0], color=colors[4], lw=0, label='5-9', marker="o"),
    mlines.Line2D([0], [0], color=colors[3], lw=0, label='10+', marker="o"),
]
ax4.legend(handles=legend_list, title='Size of Local transmission cluster', fontsize=11, title_fontsize=11, loc='upper center')

ax4.set_ylabel('Persistence of Transmission Chain \n (in days since 2023)', fontsize=12)
ax4.set_xlabel('Month of Year', fontsize=12)
ax4.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax4.text(-0.1, 1, "C", transform=ax4.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')

plt.savefig('../figures/fig_pdf/Fig4abc.pdf', dpi=300, bbox_inches='tight')


## Supp Fig  6-- trying to quantify the impact of intros on downstream cases

In [ ]:
#making decimal date from string dates adapted from stackoverflow 
def toYearFraction(date):
    def sinceEpoch(date): # returns seconds since epoch
        return time.mktime(date.timetuple())
    s = sinceEpoch

    year = date.year
    startOfThisYear = dt(year=year, month=1, day=1)
    startOfNextYear = dt(year=year+1, month=1, day=1)

    yearElapsed = s(date) - s(startOfThisYear)
    yearDuration = s(startOfNextYear) - s(startOfThisYear)
    fraction = yearElapsed/yearDuration

    return date.year + fraction


In [ ]:
def enumerate_prop_intros_time_period(tree, traitType):
    
    # we want two month time windows for the entire time period
    start_date = dt.strptime("2023-03-01", "%Y-%m-%d")
    final_date = dt.strptime("2024-12-12", "%Y-%m-%d")

    time_windows = []
    current_start = start_date
    while current_start < final_date: #this is to make sure the calculations dont over lap
        current_end = current_start + relativedelta(months=2)
        if current_end > final_date:
            current_end = final_date
        time_windows.append((current_start, current_end))
        current_start = current_end

    # Convert to decimal years
    time_windows_decimal = [
        (toYearFraction(start), toYearFraction(end))
        for (start, end) in time_windows
    ]

    traitType = "obs"
    overall_migration_events_counter = 0  # total over all windows
    output_dict = {}

    # extract all the migration events at once so that it's more efficient 
    migration_events = []
    for k in tree.Objects:
        if traitType not in k.traits:
            trait = "root"
        else:
            trait = str(k.traits[traitType])

        parent_node = k.parent

        if traitType not in parent_node.traits:
            parent_trait = "root"
        else:
            parent_trait = str(parent_node.traits[traitType])

        if (trait != parent_trait) and (parent_trait != "root"):
            migration_date = parent_node.absoluteTime #+ (k.absoluteTime - parent_node.absoluteTime) * random.uniform(0,1)
            leaf_name_set = set(parent_node.leaves)
            leaf_list = [leaf for leaf in tree.getExternal() if leaf.name in leaf_name_set]
            chain_latest_tip = max(x.absoluteTime for x in leaf_list)
            
            # save the results!
            migration_events.append({
                "parent_trait": parent_trait,
                "trait": trait,
                "migration_date": migration_date,
                "chain_latest_tip": chain_latest_tip,
                "leaf_list": leaf_list
            })

            overall_migration_events_counter += 1

    # Now loop through each time window
    for idx, (start_time, end_time) in enumerate(time_windows_decimal):
        intro_in_time_period = 0
        intro_persist = 0
        ongoing_lineage = 0
        descendents_intro_list = []
        descendents_ongoing_list = []

        for event in migration_events:
            migration_date = event["migration_date"]
            chain_latest_tip = event["chain_latest_tip"]
            leaf_list = event["leaf_list"]

            if (migration_date > start_time) and (migration_date <= end_time):
                intro_in_time_period += 1
                if chain_latest_tip >= end_time:
                    intro_persist += 1
                    descendents_intro_list.extend([x.absoluteTime for x in leaf_list if x.absoluteTime >= end_time])

            elif (migration_date < start_time) and (chain_latest_tip >= end_time):
                ongoing_lineage += 1
                descendents_ongoing_list.extend([x.absoluteTime for x in leaf_list if x.absoluteTime >= end_time])

        number_of_descendents_intro = len(descendents_intro_list)
        number_of_descendents_ongoing = len(descendents_ongoing_list)
           
        # calculate proportions
        if (intro_in_time_period + ongoing_lineage) > 0:
            prop_unique_intros = intro_in_time_period / (intro_in_time_period + ongoing_lineage)
        else:
            prop_unique_intros = None

        if (intro_persist + ongoing_lineage) > 0:
            prop_intro_persist = intro_persist / (intro_persist + ongoing_lineage)
        else:
            prop_intro_persist = None

        if (number_of_descendents_intro + number_of_descendents_ongoing) > 0:
            prop_intro_descendents = number_of_descendents_intro / (number_of_descendents_intro + number_of_descendents_ongoing)
        else:
            prop_intro_descendents = None

        output_dict[idx] = {"prop_unique_intros":prop_unique_intros, "prop_intro_persist":prop_intro_persist, "prop_intro_descendents":prop_intro_descendents,
                                             "intro_in_time_period": intro_in_time_period, "intro_persist": intro_persist, "ongoing_lineages": ongoing_lineage, "number_of_descendents_intro":number_of_descendents_intro,
                                             "number_of_descendents_ongoing":number_of_descendents_ongoing,
                                             "start_date": start_time, "end_date": end_time}

    return(output_dict)


In [ ]:
#counts all migration events and records parent and child nodes
def run_intro_time_period_counts(all_trees, traitType):
    start_time = time.time()
    with open(all_trees, "r") as infile:

        tree_counter = 0
        trees_processed = 0
        migrations_dict = {}

        for line in infile:
            if 'tree STATE_' in line:
                tree_counter += 1

                if tree_counter > burnin:
                    temp_tree = StringIO(taxa_lines + line)
                    
                    tree = bt.loadNexus(temp_tree, absoluteTime = False)
                    tree.setAbsoluteTime(2024.9467)
                    trees_processed += 1

                    # iterate through the tree and pull out all migration events
                    migrations_dict[tree_counter] = enumerate_prop_intros_time_period(tree, traitType)

    # print the amount of time this took
    total_time_seconds = time.time() - start_time
    total_time_minutes = total_time_seconds/60
    print("this took", total_time_seconds, "seconds (", total_time_minutes," minutes) to run on", trees_processed, "trees")
   
    """this will generate a multi-index dataframe from the migrations dictionary"""
    migrations_df = pd.DataFrame.from_dict({(i,j): migrations_dict[i][j] 
                           for i in migrations_dict.keys() 
                           for j in migrations_dict[i].keys()},
                       orient='index')

    migrations_df.reset_index(inplace=True)
    migrations_df.rename(columns={'level_0': 'tree_number', 'level_1': 'migration_event_number'}, inplace=True)
    
    return(migrations_df)

In [ ]:
#identify each migration jump across posterior set of trees
migrations_df = run_intro_time_period_counts(all_trees, traitType = "obs")

In [ ]:
migrations_df

In [ ]:
migrations_df["calendar_start_date"] = migrations_df.start_date.map(convert_partial_year)
migrations_df["calendar_end_date"] = migrations_df.end_date.map(convert_partial_year)
migrations_df['year_month_start'] = migrations_df['calendar_start_date'].map(convert_format_month)
migrations_df['year_month_end'] = migrations_df['calendar_end_date'].map(convert_format_month)

In [ ]:
migrations_df.head()

In [ ]:
# read in empirical mpox case data
la_mpox_cases_df = pd.read_csv("../multitree_coalescent/data/monkeypox_data.csv")
la_mpox_cases_df = la_mpox_cases_df.dropna(how = "all").dropna(axis = "columns", how = "all")
la_mpox_cases_df = la_mpox_cases_df.rename(columns= {"Unnamed: 0": "date"}); la_mpox_cases_df.head()
#format dates
weekly_cases = la_mpox_cases_df.copy()
weekly_cases.date = pd.to_datetime(weekly_cases['date'])
weekly_cases = weekly_cases.set_index("date")
weekly_cases = weekly_cases.resample("W").sum()
weekly_cases = weekly_cases.reset_index()
weekly_cases["decimal_date"] =weekly_cases["date"].map(toYearFraction)

weekly_cases.head()

#more date formatting 
la_mpox_cases_df.date = pd.to_datetime(la_mpox_cases_df['date'])
la_mpox_cases_df["decimal_date"] =la_mpox_cases_df.date.map(toYearFraction)

In [ ]:
fig,ax = plt.subplots(figsize=(10,5),facecolor='w')

# sort and drop NA
df_sorted = migrations_df.sort_values(by='start_date')
df_sorted = df_sorted.dropna(subset=['prop_unique_intros', 'prop_intro_persist', 'prop_intro_descendents', 'start_date', 'end_date'])


# group by date
summary_df = df_sorted.groupby('start_date').agg({
    'prop_unique_intros': ['mean', 'sem'],
    'prop_intro_persist': ['mean', 'sem'],
    'prop_intro_descendents': ['mean', 'sem']
})

# this is done because pandas creates a multi index column and we need it flat!
summary_df.columns = [
    'mean_unique', 'sem_unique',
    'mean_persist', 'sem_persist',
    'mean_descendents', 'sem_descendents'
]

#  95% confidence intervals
summary_df['ci_unique'] = 1.96 * summary_df['sem_unique']
summary_df['ci_persist'] = 1.96 * summary_df['sem_persist']
summary_df['ci_descendents'] = 1.96 * summary_df['sem_descendents']

# bring in the other values 
other_values =  df_sorted.groupby('start_date').first()

final_df = summary_df.merge(other_values, left_index=True, right_index=True)


ax2 = ax.twinx()

#plot the main figure with weekly cases
ax2.bar(weekly_cases.decimal_date, weekly_cases["cases"], color="#003f5c", alpha = 0.4, width = 0.019)

# Prop unique intros
# ax.scatter([start_dates+0.07, start_dates+0.07], [mean_unique, mean_unique])
# ax.plot([start_dates+0.07, start_dates+0.07], [mean_unique-ci_unique, mean_unique+ci_unique,])

for inx, row in final_df.reset_index().iterrows():
    ax.scatter(row.start_date + 0.06, row.mean_unique,  s = (row.intro_in_time_period + row.ongoing_lineages) *8, color = 'blue' )
    ax.plot([row.start_date + 0.06,row.start_date + 0.06], [row.mean_unique - (1.96 *row.sem_unique),row.mean_unique +(1.96 *row.sem_unique) ],   color = 'blue' )

    ax.scatter(row.start_date + 0.08, row.mean_persist,  s = (row.intro_persist + + row.ongoing_lineages) *8, color = 'orange')
    ax.scatter(row.start_date + 0.11, row.mean_descendents,  s = (row.number_of_descendents_intro + row.number_of_descendents_ongoing) *8, color = 'green')

    
#ax.errorbar(final_df.index+0.07, final_df.mean_unique, yerr=final_df.ci_unique, fmt='o', label='Ratio of all introductions', capsize = 6,  color='blue')
ax.plot(final_df.index+0.06, final_df.mean_unique, color='blue', alpha = 0.1)


# # Prop intros persisting
#ax.errorbar(final_df.index+0.09, final_df.mean_persist, yerr=final_df.ci_persist, fmt='o', capsize = 6,  label='Ratio of persistent introductions', color='orange')
ax.plot( final_df.index+0.08, final_df.mean_persist, alpha = 0.1, color='orange', )


# # Prop descendents
#ax.errorbar(final_df.index+0.12, final_df.mean_descendents, yerr=final_df.ci_descendents, fmt='o', capsize = 6, label='Ratio of descendents of persistent introductions', color='green')
ax.plot(final_df.index+0.11, final_df.mean_descendents, color='green', alpha = 0.1)

# # Add vertical dashed lines for window boundaries
# for start in start_dates:
#     ax.axvline(x=start, color='grey', linestyle='--', alpha=0.5)

ax.axhline(y= 0.5, color = 'black', linestyle = '--', alpha = 0.5)

xDates=['%04d-%02d-01'%(y,m) for y in range(2023,2025) for m in range(1,13)]
xDates2=['%04d-%02d-01'%(y,m) for y in range(2023,2025) for m in range(1,13)]


every=1
[ax.axvspan(bt.decimalDate(xDates2[x]),bt.decimalDate(xDates2[x])+2/float(12),facecolor='k',edgecolor='none',alpha=0.04) for x in range(0,len(xDates2),4)]
ax.set_xticks([bt.decimalDate(x)+1/24.0 for x in xDates if (int(x.split('-')[1])-1)%every==0])

ax.set_xticklabels([convertDate(x,'%Y-%m-%d','%Y') if x.split('-')[1]=='01' else convertDate(x,'%Y-%m-%d','%b') for x in xDates if (int(x.split('-')[1])-1)%every==0])
ax.tick_params(axis='x',labelsize=10,size=0)  

ax.set_ylabel('Proportion', fontsize=12)
#ax.set_title('Mean Proportions with 95% CI Over Time', fontsize=16)
ax.set_ylim(0, 1)
ax.set_xlim(2023,2025)
ax2.set_ylim(0,40)
ax2.set_ylabel("Weekly mpox cases in LAC", rotation = -90, labelpad = 12, size = 12)
ax.grid(False)
ax.legend(fontsize=10, loc = "upper right")

plt.tight_layout()
plt.show()


In [ ]:

# Function to compute mean and HPD for a single group
def compute_hpd(group):
    result = {
        'mean_unique': group['prop_unique_intros'].mean(),
        'mean_persist': group['prop_intro_persist'].mean(),
        'mean_descendents': group['prop_intro_descendents'].mean(),
    }

    hdi_unique = az.hdi(group['prop_unique_intros'].to_numpy(), hdi_prob=0.90)
    hdi_persist = az.hdi(group['prop_intro_persist'].to_numpy(), hdi_prob=0.90)
    hdi_descendents = az.hdi(group['prop_intro_descendents'].to_numpy(), hdi_prob=0.90)

    result.update({
        'hpd_unique_low': hdi_unique[0],
        'hpd_unique_high': hdi_unique[1],
        'hpd_persist_low': hdi_persist[0],
        'hpd_persist_high': hdi_persist[1],
        'hpd_descendents_low': hdi_descendents[0],
        'hpd_descendents_high': hdi_descendents[1],
    })

    return pd.Series(result)




fig,ax = plt.subplots(figsize=(10,5),facecolor='w')

# sort and drop NA
df_sorted = migrations_df.sort_values(by='start_date')
df_sorted = df_sorted.dropna(subset=['prop_unique_intros', 'prop_intro_persist', 'prop_intro_descendents', 'start_date', 'end_date'])


# Apply this per start_date group
summary_df = df_sorted.groupby('start_date').apply(compute_hpd)

# bring in the other values 
other_values =  df_sorted.groupby('start_date').first()

final_df = summary_df.merge(other_values, left_index=True, right_index=True)


ax2 = ax.twinx()

#plot the main figure with weekly cases
ax2.bar(weekly_cases.decimal_date, weekly_cases["cases"], color="#003f5c", alpha = 0.4, width = 0.019)


for inx, row in final_df.reset_index().iterrows():
    ax.scatter(row.start_date + 0.06, row.mean_unique,  s = (row.intro_in_time_period + row.ongoing_lineages) *8, color = 'blue' , alpha = 0.7)
    ax.plot([row.start_date + 0.06,row.start_date + 0.06], [row.hpd_unique_low, row.hpd_unique_high ],   color = 'blue', alpha = 0.7 )

    ax.scatter(row.start_date + 0.08, row.mean_persist,  s = (row.intro_persist + + row.ongoing_lineages) *8, color = 'orange', alpha = 0.7)
    ax.plot([row.start_date + 0.08,row.start_date + 0.08], [row.hpd_persist_low, row.hpd_persist_high ],   color = 'orange' , alpha = 0.7)

    ax.scatter(row.start_date + 0.11, row.mean_descendents,  s = (row.number_of_descendents_intro + row.number_of_descendents_ongoing) *8, color = 'green', alpha = 0.7)
    ax.plot([row.start_date + 0.11,row.start_date + 0.11], [row.hpd_descendents_low, row.hpd_descendents_high ],   color = 'green' , alpha = 0.7)


ax.axhline(y= 0.5, color = 'black', linestyle = '--', alpha = 0.5)


legend_list = [mlines.Line2D([], [], color="blue", lw=0, label='total intros/ all lineages', marker = "o"),
                mlines.Line2D([0], [0], color="orange", lw=0, label='persistent intros/ all lineages', marker = "o"),
                mlines.Line2D([0], [0], color="green", lw=0, label='Intro descendents/ all descendents', marker = "o")]

ax.legend(handles = legend_list,title='', fontsize=10, title_fontsize=12, loc='upper right')

xDates=['%04d-%02d-01'%(y,m) for y in range(2023,2025) for m in range(1,13)]
xDates2=['%04d-%02d-01'%(y,m) for y in range(2023,2025) for m in range(1,13)]


every=1
[ax.axvspan(bt.decimalDate(xDates2[x]),bt.decimalDate(xDates2[x])+2/float(12),facecolor='k',edgecolor='none',alpha=0.04) for x in range(0,len(xDates2),4)]
ax.set_xticks([bt.decimalDate(x)+1/24.0 for x in xDates if (int(x.split('-')[1])-1)%every==0])

ax.set_xticklabels([convertDate(x,'%Y-%m-%d','%Y') if x.split('-')[1]=='01' else convertDate(x,'%Y-%m-%d','%b') for x in xDates if (int(x.split('-')[1])-1)%every==0])
ax.tick_params(axis='x',labelsize=10,size=0)  

ax.set_ylabel('Proportion', fontsize=12)
#ax.set_title('Mean Proportions with 95% CI Over Time', fontsize=16)
ax.set_ylim(0, 1)
ax.set_xlim(2023,2025)
ax2.set_ylim(0,40)
ax2.set_ylabel("Weekly mpox cases in LAC", rotation = -90, labelpad = 12, size = 12)
ax.grid(False)
#ax.legend(fontsize=10, loc = "upper right")

plt.tight_layout()
#plt.savefig('../figures/two_month_intro_quant.png',dpi=300,bbox_inches='tight')
plt.show()



## Supp fig S5 calculate the number of transitions over the tree set. 

In [ ]:
start_time = time.time()

mig = return_proportions_dataframe(migrations_df, "year-week")

total_time_seconds = time.time() - start_time
total_time_minutes = total_time_seconds/60
print(total_time_minutes)

mig.head()

In [ ]:
mig.reset_index(inplace = True, drop = True)


In [ ]:
mig.head()

In [ ]:
mig_to_export = mig[["year-week", "tree_number", "total_transitions", "transitions_in_time_interval"]]; mig_to_export.head()

In [ ]:
mig_to_export.to_csv("mpox_la_introductions_over_time.csv", sep = ",")

In [ ]:
# calculate HPDs
grouped = mig_to_export.groupby("year-week")["transitions_in_time_interval"]

summary = grouped.agg(['median', 'sem'])

hpd_50 = grouped.apply(lambda s: az.hdi(s.to_numpy(), hdi_prob=0.50))
hpd_95 = grouped.apply(lambda s: az.hdi(s.to_numpy(), hdi_prob=0.95))

stats = pd.DataFrame({
    'median': summary['median'],
    'sem': summary['sem'],
    'hpd50_lo': hpd_50.apply(lambda x: x[0]),
    'hpd50_hi': hpd_50.apply(lambda x: x[1]),
    'hpd95_lo': hpd_95.apply(lambda x: x[0]),
    'hpd95_hi': hpd_95.apply(lambda x: x[1]),
})

print(stats)

In [ ]:
stats.reset_index(inplace= True)

In [ ]:
stats.to_csv("mpox_la_introductions_stats.csv", sep = ",")

In [ ]:
stats

## Plot Figure S5!!

In [ ]:


## composite figure S5
fig = plt.figure(figsize=(16,16),facecolor='w')


gs = GridSpec(2, 1, height_ratios=[1, 1],  hspace=0.2)    

ax = fig.add_subplot(gs[0])

# set blank white face for background    
ax.set_facecolor('white')
# remove grid 
#ax.grid(False)

for index, values in migrations_for_plot.iterrows():
   # print(index)
    clust = index + 10
    if values.size_of_chain <2:
        col = colors[5]
    elif (values.size_of_chain >1) & (values.size_of_chain <5):
        col = colors[1]
    elif (values.size_of_chain >4) & (values.size_of_chain <11):
        col = colors[4]
    else:
        col = colors[3]
    linewidth = 3
    persist_days = decimal_to_days(values.chain_latest_tip - values.date)
    radius = np.sqrt(values.size_of_chain/np.pi)*300.0
    
    ax.scatter(values.date,persist_days,s=100,facecolor=col,edgecolor='k',lw=2,zorder=200) ## add big circle at base of tree to indicate origin
    ax.plot([values.date, values.chain_latest_tip], [persist_days, persist_days], color=col, linewidth=1, linestyle = ":")

ax.set_ylabel('Persistence time (circles in days)', fontsize=20)
   
fc = colors[0]
ec = colors[0]
ax2 = ax.twinx()

ax2.plot(test_mig.decimal_date,test_mig["mean_mig_linear"],color=fc,ls='--',lw=2)

#ax.scatter(caseDates,[0.0]*len(caseDates),alpha=0.2,s=200,marker='|',lw=3,facecolor='k',zorder=100)
ax2.fill_between(test_mig.decimal_date,test_mig.lower_hpd_linear_50,test_mig.upper_hpd_linear_50,alpha=0.05,facecolor=fc,edgecolor=ec,zorder=1000)
ax2.plot(test_mig.decimal_date,test_mig.lower_hpd_linear_50,color=fc,lw=1,zorder=1000)
ax2.plot(test_mig.decimal_date,test_mig.upper_hpd_linear_50,color=fc,lw=1,zorder=1000)
ax2.set_ylim(0,30)



fc = colors[2]
ec = colors[2]
ax3 = ax.twinx()

ax3.plot(test_ne.decimal_date,test_ne["mean_Ne_linear_MA"],color=fc,ls='--',lw=2)

#ax.scatter(caseDates,[0.0]*len(caseDates),alpha=0.2,s=200,marker='|',lw=3,facecolor='k',zorder=100)


ax3.fill_between(test_ne.decimal_date,test_ne.lower_hpd_linear_50_MA,test_ne.upper_hpd_linear_50_MA,alpha=0.05,facecolor=fc,edgecolor=ec,zorder=1000)
ax3.plot(test_ne.decimal_date,test_ne.lower_hpd_linear_50_MA,color=fc,lw=1,zorder=1000)
ax3.plot(test_ne.decimal_date,test_ne.upper_hpd_linear_50_MA,color=fc,lw=1,zorder=1000)
ax3.set_ylim(0,30)
ax3.set_ylabel('Importation Rate (Yellow) & \n Effective Population Size (Purple)', fontsize=18, rotation = -90, labelpad=40)



legend_list = [mlines.Line2D([0], [0], color=colors[5], lw=4, label='Singletons'),
                mlines.Line2D([0], [0], color=colors[1], lw=4, label='2-4'),
                mlines.Line2D([0], [0], color=colors[4], lw=4, label='5-9'),
                mlines.Line2D([0], [0], color=colors[3], lw=4, label='10+')]
ax.legend(handles=legend_list, title='Size of Local transmission cluster', fontsize=16, title_fontsize=16, loc='upper right')

xDates=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,13)]
xDates2=['%04d-%02d-01'%(y,m) for y in range(2022,2025) for m in range(1,13)]


every=1
[ax.axvspan(bt.decimalDate(xDates2[x]),bt.decimalDate(xDates2[x])+1/float(12),facecolor='k',edgecolor='none',alpha=0.04) for x in range(0,len(xDates2),2)]
ax.set_xticks([bt.decimalDate(x)+1/24.0 for x in xDates if (int(x.split('-')[1])-1)%every==0])

ax.set_xticklabels([convertDate(x,'%Y-%m-%d','%Y') if x.split('-')[1]=='01' else convertDate(x,'%Y-%m-%d','%b') for x in xDates if (int(x.split('-')[1])-1)%every==0])
ax.tick_params(axis='x',labelsize=15,size=0)  
ax.tick_params(axis='y',size=0, labelsize=20)
ax.yaxis.tick_left()
[ax2.spines[loc].set_visible(False) for loc in ['top',]]
[ax.spines[loc].set_visible(False) for loc in ['top','right']]
#ax.set_yticklabels([])
ax.set_ylim(0,500)
ax.set_xlim(2022,2025.1)

# Rotate x-axis labels
for lbl in ax.get_xticklabels():
    lbl.set_rotation(45)
    lbl.set_ha('right')

ax.xaxis.set_tick_params(which='both', top=False, bottom=True, labelbottom=True)
ax.yaxis.set_tick_params(which='both', right=False, left=True, labelleft=True)


### now plot the weekly importations!

ax_4 = fig.add_subplot(gs[1])

grouped = mig.groupby("year-week")["transitions_in_time_interval"]
stats = grouped.agg(['median'])

def safe_hdi(x, prob):
    hdi = np.sort(az.hdi(np.asarray(x), hdi_prob=prob))
    return hdi

hpd_90 = grouped.apply(lambda s: safe_hdi(s, 0.9))

stats['hpd90_lo'] = hpd_90.apply(lambda x: x[0])
stats['hpd90_hi'] = hpd_90.apply(lambda x: x[1])
stats = stats.sort_index()

# format dates
dates = pd.to_datetime(stats.index)

# Compute positive errors but nonsymettrically

yerr_90 = np.abs(np.vstack([
    stats['median'] - stats['hpd90_lo'],
    stats['hpd90_hi'] - stats['median']
]))


plot_dates = pd.to_datetime(stats.index)

ax_4.errorbar(
    plot_dates, stats['median'], yerr=yerr_90,
    fmt='o-', ecolor='black', capsize=4, elinewidth=2.5, linewidth=2, alpha=0.4,
)

ax_4.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1, interval=1))
ax_4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%b'))

# rotate labels 
for lbl in ax_4.get_xticklabels():
    lbl.set_rotation(45)
    lbl.set_ha('right')
    
# format
ax_4.set_title("Transitions in Time Interval", size = 20)
ax_4.set_xlabel("Month", size = 18)
ax_4.set_ylabel("Transitions in Time Interval", size = 18)
ax_4.tick_params(axis='x',labelsize=15,size=0)  
ax_4.tick_params(axis='y',size=0, labelsize=20)
ax_4.grid(which='major', axis='x', color='gray', linestyle='--', alpha=0.6)



ax.text(-0.1, 1, "A", transform=ax.transAxes, fontsize=25, fontweight='bold', va='top', ha='right')  
ax_4.text(-0.1, 1, "B", transform=ax_4.transAxes, fontsize=25, fontweight='bold', va='top', ha='right')  


plt.savefig('../figures/mpox_la_introduction_persistence_with_ne.png',dpi=300,bbox_inches='tight')
